# Fitting Vs Validation

Generate graphs displaying the learning and validation curves for each metric per client.

Assuming that this notebook will be used after having completed a training run using the framework, it will fetch the fitting and validation metrics from the server-side Minio datastore. For this to work, place a file named `scenario.json` in the same directory as this notebook containing the scenario that has been used to train the model with.

Set the values inherent to your setup in the `Constants` section.

### Imports

In [ ]:
import json
import matplotlib.pyplot as plt
from minio import Minio

### Constants

If the INFERENCE_DATASET(.parquet) is not found in this folder, the notebook will try to download it from Kaggle according to this format: [username]-[dataset]

In [ ]:
MINIO_SERVER_ENDPOINT = "[server-ip]:9000" # change with your server's IP
MINIO_SERVER_ACCESS_KEY = "username" # change with your server's datastore credentials
MINIO_SERVER_SECRET_KEY = "password" # change with your server's datastore credentials
MINIO_SERVER_METRICS_BUCKET = "metrics"

### Load scenario

In [ ]:
with open("scenario.json", "r") as f:
    scenario = json.load(f)

scenario_name = scenario["name"]

### Download metrics

In [ ]:
metrics_path = f"{scenario_name}_fitted_validated_metrics.json"

minio_client = Minio(MINIO_SERVER_ENDPOINT, access_key=MINIO_SERVER_ACCESS_KEY, secret_key=MINIO_SERVER_SECRET_KEY, secure=False)
minio_client.fget_object(MINIO_SERVER_METRICS_BUCKET, metrics_path, metrics_path)

### Fitting Vs Validation

In [ ]:
def plot_knit_metrics(history):
    """
    Stitches metrics across rounds into continuous plots per client.
    Shows graphs per metric per client.
    Adds vertical dashed lines to mark round boundaries.
    """

    # Collect data per client across rounds
    client_histories = {}
    metrics_recorded = list(next(iter(history.values()))[0].keys())
    epochs_per_round = len(next(iter(next(iter(history.values()))[0].values())))
    n_metrics = len(metrics_recorded)
    round_breaks = range(epochs_per_round, epochs_per_round*len(history), epochs_per_round)

    for client_list in history.values():
        for i in range(len(client_list)):
            client_data = client_list[i]
            client_name = i
            
            if client_name not in client_histories:
                client_histories[client_name] = client_data
            else:
                # Append metrics
                for metric_name in client_data.keys():
                    client_histories[client_name][metric_name].extend(client_data[metric_name])

    # Plot per client
    for client_name, metrics in client_histories.items():
        epochs = range(1, epochs_per_round * len(history) + 1)
        
        for i in range(n_metrics // 2):
            train_metric_name = metrics_recorded[i]
            val_metric_name = metrics_recorded[i + n_metrics//2]
            
            plt.figure(figsize=(8, 4))
            plt.plot(epochs, metrics[train_metric_name], label="Training " + train_metric_name)
            plt.plot(epochs, metrics[val_metric_name], label="Validation " + train_metric_name)
            for rb in round_breaks:
                plt.axvline(rb, color="green", linestyle="--", alpha=0.6)
            plt.title(f"Client {client_name} - " + train_metric_name)
            plt.xlabel("Epochs (stitched across rounds)")
            plt.ylabel(train_metric_name)
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.show()


with open(metrics_path) as f:
    history = json.load(f)
    
plot_knit_metrics(history)